In [21]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.transform import from_origin
from rasterio.windows import from_bounds
from rasterio.vrt import WarpedVRT

def fast_zonal_stats(
    raster_path,
    zones_path,
    zone_id_field,                  # column in zones for ID (string, int, etc.)
    target_res=None,                # None (native) or float or (xres, yres)
    all_touched=False,              # rasterize option
    resampling="nearest",           # nearest/bilinear/cubic when resampling
    stats=("count", "sum", "mean", "min", "max", "std"),
    filter_empty=True,              # drop zones with 0 overlapping pixels
):
    """
    Fast zonal stats in pure Python using rasterio + numpy.
    - Dynamically set resolution with target_res (uses WarpedVRT)
    - Processing extent cropped to zones extent
    - Returns a pandas DataFrame with one row per zone_id
    """

    resampling_map = {
        "nearest": Resampling.nearest,
        "bilinear": Resampling.bilinear,
        "cubic": Resampling.cubic,
    }
    if resampling not in resampling_map:
        raise ValueError("resampling must be one of: nearest, bilinear, cubic")

    # Open raster
    with rasterio.open(raster_path) as src:
        # Read zones and reproject to raster CRS
        zones = gpd.read_file(zones_path)
        if zones.crs is None:
            raise ValueError("Zones CRS is undefined.")
        if zones.empty:
            raise ValueError("Zones layer is empty.")
        if zones.geometry.is_empty.all():
            raise ValueError("All geometries in zones are empty.")
        zones = zones.to_crs(src.crs)

        # Keep only zones with geometry
        zones = zones[~zones.geometry.is_empty].copy()
        if zone_id_field not in zones.columns:
            raise ValueError(f"zone_id_field '{zone_id_field}' not found in zones.")

        # Compute total bounds in raster CRS, crop raster reads to these bounds
        left, bottom, right, top = zones.total_bounds

        # Build dense integer IDs for fast bincount aggregation
        # (0 is reserved for background)
        unique_ids = pd.Index(zones[zone_id_field].unique())
        id_to_dense = {val: i + 1 for i, val in enumerate(unique_ids)}
        dense_to_id = {v: k for k, v in id_to_dense.items()}
        zones["__dense__"] = zones[zone_id_field].map(id_to_dense).astype(np.int32)

        # Decide whether to use native grid (window) or a WarpedVRT at a new resolution
        resampling_enum = resampling_map[resampling]

        def _read_native_window():
            win = from_bounds(left, bottom, right, top, transform=src.transform)
            win = win.round_offsets().round_lengths()
            data = src.read(1, window=win, boundless=True)
            nodata = src.nodata
            transform = rasterio.windows.transform(win, src.transform)
            return data, nodata, transform, (win.height, win.width)

        def _read_resampled_vrt():
            # Target transform + size from desired resolution and zones extent
            if np.isscalar(target_res):
                xres = float(target_res)
                yres = float(target_res)
            else:
                xres, yres = float(target_res[0]), float(target_res[1])

            width = int(np.ceil((right - left) / xres))
            height = int(np.ceil((top - bottom) / yres))
            transform = from_origin(left, top, xres, yres)

            with WarpedVRT(
                src,
                crs=src.crs,             # keep CRS, just resample
                transform=transform,
                width=width,
                height=height,
                resampling=resampling_enum,
            ) as vrt:
                data = vrt.read(1)
                nodata = vrt.nodata
                return data, nodata, vrt.transform, (height, width)

        if target_res is None:
            data, nodata, out_transform, (h, w) = _read_native_window()
        else:
            data, nodata, out_transform, (h, w) = _read_resampled_vrt()

        # Rasterize zones onto the same grid
        shapes = [(geom, did) for geom, did in zip(zones.geometry, zones["__dense__"])]
        zone_arr = rasterize(
            shapes=shapes,
            out_shape=(h, w),
            transform=out_transform,
            fill=0,
            all_touched=all_touched,
            dtype="int32",
        )

        # Build validity mask from nodata and compute stats using vectorized ops
        a = data
        if nodata is None:
            valid = ~np.isnan(a) if np.issubdtype(a.dtype, np.floating) else np.ones(a.shape, dtype=bool)
        else:
            if np.issubdtype(a.dtype, np.floating) and np.isnan(nodata):
                valid = ~np.isnan(a)
            else:
                valid = a != nodata

        labels = zone_arr
        mask = (labels > 0) & valid
        if not np.any(mask):
            # No overlap
            out = pd.DataFrame({zone_id_field: unique_ids, "count": 0})
            for s in stats:
                if s not in out.columns and s != "count":
                    out[s] = np.nan
            return out

        lbl = labels[mask].astype(np.int64)
        vals = a[mask].astype(np.float64)

        max_label = int(lbl.max())
        size = max_label + 1

        results = {"__dense__": np.arange(size, dtype=np.int64)}

        # counts & sums
        counts = np.bincount(lbl, minlength=size)
        sums = np.bincount(lbl, weights=vals, minlength=size)
        if "count" in stats:
            results["count"] = counts
        if "sum" in stats:
            results["sum"] = sums
        if "mean" in stats or "std" in stats:
            with np.errstate(invalid="ignore", divide="ignore"):
                means = sums / counts
        if "mean" in stats:
            results["mean"] = means

        # min/max via in-place reductions
        if "min" in stats:
            mins = np.full(size, np.inf, dtype=np.float64)
            np.minimum.at(mins, lbl, vals)
            mins[counts == 0] = np.nan
            results["min"] = mins
        if "max" in stats:
            maxs = np.full(size, -np.inf, dtype=np.float64)
            np.maximum.at(maxs, lbl, vals)
            maxs[counts == 0] = np.nan
            results["max"] = maxs

        if "std" in stats:
            sumsq = np.bincount(lbl, weights=vals * vals, minlength=size)
            with np.errstate(invalid="ignore", divide="ignore"):
                var = sumsq / counts - means * means
            var[counts == 0] = np.nan
            std = np.sqrt(var)
            results["std"] = std

        # Build DataFrame for dense labels present in zone_arr
        df = pd.DataFrame(results)
        # Keep only labels that actually appear in the rasterized result
        present = counts > 0
        df = df[present].copy()

        # Map back to original zone_id
        df[zone_id_field] = df["__dense__"].map(lambda d: dense_to_id.get(int(d)))
        df = df.drop(columns=["__dense__"])

        # Reorder columns
        cols = [zone_id_field] + [c for c in ["count", "sum", "mean", "min", "max", "std"] if c in df.columns]
        df = df[cols]

        if filter_empty:
            df = df[df["count"] > 0] if "count" in df.columns else df

        return df

ModuleNotFoundError: No module named 'rasterio'

In [19]:
rast = 'O:/LAB/COR/Geospatial_Library_Projects/StreamCat/LandscapeRasters/QAComplete/bfi.tif'
zones = 'G:/NHDPlusV21/NHDPlusCA/NHDPlus18/NHDPlusCatchment/Catchment.shp'

results = fast_zonal_stats_vector(
    rast, zones, zone_field="GRIDCODE",
    res_mode="fixed", fixed_res=(30,30),   
    resampling="average",               
    all_touched=False,
    stats=("count","mean","min","max","std"),
    num_threads=25
)

RuntimeError: '-tr xres yres' or '-ts xsize ysize' is required.

In [ ]:
import pandas as pd

df = pd.DataFrame.from_dict(results, orient="index")
df.index.name = "zone_id"
print(df.head(10))